In [3]:
from selenium import webdriver
from selenium.webdriver import Chrome, ChromeOptions
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import  WebDriverWait
import pandas as pd 
from selenium.webdriver.common.action_chains import ActionChains
import time, threading, os, requests, ast,json,re
from datetime import datetime


def GetAuctionDetails(driver):
    wait = WebDriverWait(driver, 15)

    try:
        # elements
        auction_name = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "h1.cc-auction-overview__title"))
        ).text.strip()

        auction_time_raw = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".cc-auction-overview__when dd"))
        ).text.strip()

        auction_type = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".cc-key-pair__definition"))
        ).text.strip()

        center_location = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".cc-auction-overview__where .u-line-height-1"))
        ).text.strip()



        start_part, end_part = auction_time_raw.split(" - ")

        current_year = datetime.now().year


        start_clean = re.sub(r"^\w+\s+", "", start_part).replace(" at ", " ")
        start_dt = datetime.strptime(f"{start_clean} {current_year}", "%d %B %I:%M%p %Y")

        end_clean = re.sub(r"^\w+\s+", "", end_part).replace(" at ", " ")
        end_dt = datetime.strptime(f"{end_clean} {current_year}", "%d %b %I:%M%p %Y")

        auction_data = {
            "auction_name": auction_name,
            "auction_type": auction_type,
            "start_date": start_dt.strftime("%Y-%m-%d"),
            "start_time": start_dt.strftime("%H:%M"),
            "end_date": end_dt.strftime("%Y-%m-%d"),
            "end_time": end_dt.strftime("%H:%M"),
            "center_location": center_location
        }

        with open("database.json", "w", encoding="utf-8") as f:
            json.dump(auction_data, f, indent=4)

        print("✅ Auction details saved")
        print(json.dumps(auction_data, indent=4))

        return auction_data

    except Exception as e:
        print("❌ Error in GetAuctionDetails:", e)
        return None

def scarpe(id):
    path = f"https://www.wilsonsauctions.com/auctions/{id}"
    options = ChromeOptions()
    options.headless = True
    service = Service(ChromeDriverManager().install())
    driver = Chrome(service=service, options=options)
    driver.get(path)
    driver.maximize_window()

    wait = WebDriverWait(driver, 10)
    try:
        cookie_btn = wait.until(EC.element_to_be_clickable((By.ID, 'cookiescript_accept')))
        cookie_btn.click()
  
    except:
        print("No cookies button found")


    try:
        login_btn = wait.until(EC.element_to_be_clickable((By.ID, 'gtm-sign-in-sign-up')))
        login_btn.click()
    
    except:
        print("Login button not found")


    try:
        username_input = wait.until(EC.presence_of_element_located((By.ID, 'email')))
        username_input.send_keys("fourbrotherstrading@icloud.com")
        next_btn = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[@class='c-button']")))
        next_btn.click()

    except:
        print("Username input not found")

    try:
        password_input = wait.until(EC.visibility_of_element_located((By.ID, 'password')))
        password_input.clear()
        password_input.send_keys("Muhssan7865@")  
        login_btn2 = wait.until(
            EC.element_to_be_clickable((By.XPATH, "//button[@class='c-button' and @data-label='log-in']"))
        )
        login_btn2.click()

    except Exception as e:
        print("Password input not found or error:", e)
    GetAuctionDetails(driver)
    try:
        first_lot = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, "//ul[@class='cc-cards +list +results u-marg-top u-pad-top']//li[1]//a")
            )
        )
        lot_url = first_lot.get_attribute("href")
        print(f"Opening first lot: {lot_url}")
        driver.execute_script("arguments[0].click();", first_lot)
    except Exception as e:
        print("Could not open first lot:", e)
        
    folder_name = "html"
    os.makedirs(folder_name, exist_ok=True)

    lot_number = 1

    while True:
        try:
            get_Reg = wait.until(
                EC.presence_of_element_located(
                    (By.XPATH, "//div[@class='cc-reg-plate__end']//span")
                )
            )
            reg_number = get_Reg.text.strip()
            print(f"🚗 Registration Number: {reg_number}")

            try:
                inspect_tab = wait.until(
                    EC.element_to_be_clickable((
                        By.XPATH,
                        "//button[contains(., 'Inspection')] | //a[contains(., 'Inspection')]"
                    ))
                )
                driver.execute_script("arguments[0].click();", inspect_tab)
                time.sleep(2)
            except:
                pass

            try:
                expands = driver.find_elements(
                    By.XPATH, "//button[contains(@class,'accordion')]"
                )
                for e in expands:
                    driver.execute_script("arguments[0].click();", e)
                    time.sleep(0.3)
            except:
                pass

            with open(f"{folder_name}/{reg_number}.html", "w", encoding="utf-8") as f:
                f.write(driver.page_source)

            print(f"✅ Saved inspection HTML: {reg_number}.html")



            next_btn = wait.until(
                EC.element_to_be_clickable((
                    By.XPATH,
                    "//button[.//span[contains(normalize-space(), 'Next')]]"
                ))
            )
            print(next_btn)
            driver.execute_script("arguments[0].click();", next_btn)
            time.sleep(2)

            lot_number += 1

        except Exception as e:
            print("🛑 No more lots:", e)
            break

scarpe("fleet-finance-car-auction-2657")
        

❌ Error in GetAuctionDetails: not enough values to unpack (expected 2, got 1)
Opening first lot: https://www.wilsonsauctions.com/auctions/fleet-finance-car-auction-2657/lots/238044
🚗 Registration Number: WP72GYF
✅ Saved inspection HTML: WP72GYF.html
<selenium.webdriver.remote.webelement.WebElement (session="a74b6897b36a41cf38b0a430b2d479a4", element="f.814C3F145AEC62CFBFA302A5A7DEF615.d.52B94F25A7640D8E5163D20E697EE630.e.215")>
🚗 Registration Number: CL73HYY
✅ Saved inspection HTML: CL73HYY.html
<selenium.webdriver.remote.webelement.WebElement (session="a74b6897b36a41cf38b0a430b2d479a4", element="f.814C3F145AEC62CFBFA302A5A7DEF615.d.52B94F25A7640D8E5163D20E697EE630.e.220")>
🚗 Registration Number: CV74EOJ
✅ Saved inspection HTML: CV74EOJ.html
<selenium.webdriver.remote.webelement.WebElement (session="a74b6897b36a41cf38b0a430b2d479a4", element="f.814C3F145AEC62CFBFA302A5A7DEF615.d.52B94F25A7640D8E5163D20E697EE630.e.225")>
🚗 Registration Number: CA21OKG
✅ Saved inspection HTML: CA21OKG.ht

In [23]:
import os,re,json
import csv
from bs4 import BeautifulSoup
from datetime import datetime
with open(r"D:\bots\headers.json", "r", encoding="utf-8") as f:
    header_map = json.load(f)
headers = [header_map[k] for k in sorted(header_map, key=int)]

with open("database.json","r",encoding="utf-8") as f:
    database = json.load(f)

def get_base_folder_info():
    folder_name = os.path.basename(os.getcwd())

    parts = folder_name.split("-")
    if not parts or not parts[0].isdigit():
        return None, None

    sheet_id = parts[0]
    name_parts = parts[1:]
    if name_parts and name_parts[0].isdigit():
        name_parts = name_parts[1:]

    auction_name = "-".join(name_parts).strip()


    return sheet_id, auction_name


def yearGetter(val):
    if not val:
        return ""
    match = re.search(r"(\d{4})", val)
    if match:
        return match.group(1)
    return ""

def gettitle(soup):
    title_el = soup.find("h1", class_="c-heading +h1")
    title_text = title_el.get_text(strip=True) if title_el else "No Title Found"
    return title_text


def extract_make_model(title):
    if not title:
        return "", ""

    parts = title.split()

    make = parts[0]
    model = " ".join(parts[1:3])

    return make, model

def getDeraivative(soup):
    p = soup.find("section", class_="cc-lot-summary__desc")
    if not p:
        return ""

    text = p.find("p").get_text(strip=True)
    main = text.split("/")[0].strip()
    parts = main.split()
    derivative = " ".join(parts[1:])  

    return derivative

def extract_key_value_details(soup):
    data = {}

    rows = soup.select("dl.cc-key-pairs .cc-key-pair")

    for row in rows:
        key = row.find("dt")
        value = row.find("dd")

        if key and value:
            k = key.get_text(strip=True)
            v = value.get_text(strip=True)
            data[k] = v

    return data

def extract_images(soup):
    images = []

    for img in soup.select(".swiper-slide img"):
        url = img.get("src")  
        if url:
            images.append(url)

  
    images = list(dict.fromkeys(images))


    return ",".join(images)


def clean_auction_type(text):
    if not text:
        return ""

    text = text.replace("\n", " ").strip()
    text = text.replace("Online", "").replace("Learn more", "").strip()
    text = " ".join(text.split())

    return text
def clean_value(val):
    if not val:
        return ""

    val = str(val).strip()

    if val in ["-", "—", "–", "N―"]:
        return ""

    return val

def engine_size_to_liter(val):
    if not val:
        return ""
    
    match = re.search(r"(\d+)", val.replace(',', ''))
    if not match:
        return ""
    
    cc = int(match.group(1))
    liters = cc / 1000  
    return str(round(liters, 1)) 


def clean_odometer(val):
    if not val:
        return ""
    val = str(val).strip()
    if val in ["-", "—", "–", "N―"]:
        return ""
    
    digits = "".join(c for c in val if c.isdigit())
    return digits


def get_lot_number(soup):
    el = soup.find("div", class_="cc-lot-info")
    if not el:
        return ""

    text = el.get_text(" ", strip=True)

    match = re.search(r"LOT\s+(\d+)", text, re.I)
    return match.group(1) if match else ""



def extract_manual_keys():
    folder = "html"
    output_file = "Data_Wilson.csv"

    all_rows = []

    for file in os.listdir(folder):
        if file.endswith(".html"):
            file_path = os.path.join(folder, file)
            with open(file_path, "r", encoding="utf-8") as f:
                html_content = f.read()
                soup = BeautifulSoup(html_content, "html.parser")

            row = {}

            reg_el = soup.find("div", class_="cc-reg-plate__end")
            regspan = reg_el.find("span") if reg_el else None
            reg = regspan.get_text(strip=True) if regspan else ""

            pattern = re.compile(r'^[A-Z]{1,3}[0-9]{1,3}[A-Z]{1,3}$', re.I)
            if not pattern.match(reg):
                print(f"❌ Not valid: {reg} → Deleting file {file}")
                os.remove(file_path)
                continue
            else:
                result = {}
                lot_loc_el = soup.find("dt", string=lambda x: x and "Lot location" in x)
                if lot_loc_el:
                    dd = lot_loc_el.find_next_sibling("dd")
                    if dd:
 
                        a_tag = dd.find("a")
                        result["location"] = a_tag.text.strip() if a_tag else dd.text.strip()
                    else:
                        result["location"] = ""
                else:
                    result["location"] = ""

                vat_el = soup.find("dt", string=lambda x: x and "Vat status" in x)
                if vat_el:
                    dd = vat_el.find_next_sibling("dd")
                    result["vat_status"] = dd.text.strip() if dd else ""
                else:
                    result["vat_status"] = ""
                    
                btn = soup.find("button", class_="c-button +icon +nama-button +nama-2 +icon")
                grade = ""
                if btn:
                    grade = btn.find("span", class_="c-button__text").get_text(strip=True)


                
                title =gettitle(soup)
                make, model = extract_make_model(title)
                derivative = getDeraivative(soup)
                sheet_id, auction_name = get_base_folder_info()
                lot_number = get_lot_number(soup)
                auctionName = database.get("auction_name", "")
                start_date = database.get("auction_date", "")
                start_time = database.get("auction_time", "")
                centerName = database.get("center_name", "")
                details = extract_key_value_details(soup)
                year = yearGetter(details.get("First reg", ""))
                Images= extract_images(soup)
    
                
                row[header_map['1']] = auctionName or ""
                row[header_map['2']] = sheet_id or ""
                row[header_map['3']] =  "Wilsons Auctions"
                row[header_map['4']] = title
                row[header_map['5']] = reg
                row[header_map['6']] = make
                row[header_map['7']] = model
                row[header_map['8']] = derivative
                row[header_map['9']] = lot_number or ""
                row[header_map['17']] = year or ""
                row[header_map['10']] = clean_value(details.get("Type"))
                row[header_map['24']] = clean_value(details.get("Vendor"))
                row[header_map['41']] = clean_value(details.get("Doors"))
                row[header_map['12']] = clean_value(details.get("Transmission"))
                row[header_map['33']] = clean_value(details.get("Colour"))
                row[header_map['11']] = clean_value(details.get("Fuel"))
                row[header_map['26']] = engine_size_to_liter(clean_value(details.get("Engine size")))
                row[header_map['35']] = clean_value(details.get("CAP clean"))
                row[header_map['37']] = clean_value(details.get("CAP average"))
                row[header_map['16']] = clean_value(details.get("First reg"))
                row[header_map['21']] = clean_value(details.get("MOT"))
                row[header_map['18']] = clean_odometer(clean_value(details.get("Odometer")))
                row[header_map['19']] = clean_value(details.get("Mileage checked"))
                row[header_map['22']] = clean_value(details.get("Service history"))
                row[header_map['20']] = clean_value(details.get("V5 Location"))
                row[header_map['34']] = clean_value(details.get("Master key"))
                row[header_map['25']] = clean_value(details.get("Number of former keepers"))
                row[header_map['23']] = result["vat_status"] or ""
                row[header_map['13']] = centerName or ""
                row[header_map['29']] = Images or ""


        
                
                row[header_map['40']] = grade or ""
                row[header_map['14']] = start_date or ""
                row[header_map['15']] = start_time or ""
               
                row[header_map['48']] = "Online Auction" or ""
                row[header_map['49']] = details.get("Non-Runner", "") or ""
               
      

            all_rows.append(row)
           


    with open(output_file, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=headers)
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\n✔ CSV Generated: {output_file}")


extract_manual_keys()



✔ CSV Generated: Data_Wilson.csv
